# Cross-validation for KC house price regression

Uses the local `day7/kc_house_data.csv` dataset to predict house prices in USD with 18 numeric features. ID and sale date are excluded. Five shuffled folds compare linear regression and a decision tree. RMSE is expressed in USD.

In [1]:
# CROSS VALIDATION FOR REGRESSION - FULL DEMO


import numpy as np

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# 1. LOAD DATASET

# Support launching from the project root or the notebook directory.
candidates = [Path("day7/kc_house_data.csv"), Path("../day7/kc_house_data.csv")]
data_path = next((path.resolve() for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Cannot find day7/kc_house_data.csv; run from spark or spark/day8.")

data = pd.read_csv(data_path)
# Exclude incomplete modeling rows (two in the supplied CSV).
model_columns = data.columns.difference(["id", "date"])
original_rows = len(data)
data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=model_columns)
print("Incomplete rows excluded:", original_rows - len(data))
# Price is the target; ID is an identifier and date is not a numeric feature.
features = data.drop(columns=["id", "date", "price"])
X = features.to_numpy(dtype=float)
y = data["price"].to_numpy(dtype=float)
assert len(X) > 5 and X.shape[0] == len(y)
assert np.isfinite(X).all() and np.isfinite(y).all(), "Data must be numeric and finite."
print("Dataset:", data_path)
print("Target: price (USD)")
print("Features:", list(features.columns))

print("\nDataset Info")
print("Shape:", X.shape)
print("\n" + "="*50)


# 2. TRAIN-TEST SPLIT (BASELINE)

print("\n1. TRAIN-TEST SPLIT (BASELINE)")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

baseline_r2 = model.score(X_test, y_test)

print("R2 Score (Baseline):", round(baseline_r2, 4))
print("\n" + "="*50)


# 3. INSTABILITY DEMO
print("\n2. INSTABILITY OF TRAIN-TEST SPLIT")


for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i)
    model.fit(X_train, y_train)
    r2 = model.score(X_test, y_test)
    print(f"Run {i+1}: R2 = {round(r2, 4)}")

print("\n" + "="*50)


# 4. K-FOLD CROSS VALIDATION (R2)

print("\n3. K-FOLD CROSS VALIDATION (R2)")


kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')

print("R2 Scores:", np.round(r2_scores, 4))
print("Mean R2:", round(r2_scores.mean(), 4))

print("\n" + "="*50)

# 5. CROSS VALIDATION WITH MSE & RMSE

print("\n4. CROSS VALIDATION (MSE & RMSE)")


mse_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')

rmse_scores = np.sqrt(-mse_scores)

print("MSE Scores (USD squared):", np.round(-mse_scores, 2))
print("RMSE Scores (USD):", np.round(rmse_scores, 4))
print("Mean RMSE (USD):", round(rmse_scores.mean(), 4))

print("\n" + "="*50)

# 6. SINGLE MODEL ASSESSMENT

print("\n5. SINGLE MODEL EVALUATION")


print("Mean R2:", round(r2_scores.mean(), 4))
print("Std Dev:", round(r2_scores.std(), 4))

print("\n" + "="*50)


# 7. COMPARING TWO MODELS

print("\n6. MODEL COMPARISON")


model1 = LinearRegression()
model2 = DecisionTreeRegressor(random_state=42)

r2_1 = cross_val_score(model1, X, y, cv=kf, scoring='r2')
r2_2 = cross_val_score(model2, X, y, cv=kf, scoring='r2')

print("Linear Regression Mean R2:", round(r2_1.mean(), 4))
print("Decision Tree Mean R2:", round(r2_2.mean(), 4))

print("\n" + "="*50)


# 8. PAIRED K-FOLD COMPARISON

print("\n7. PAIRED K-FOLD COMPARISON")
print("----------------------------")

scores1 = []
scores2 = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model1.fit(X_train, y_train)
    model2.fit(X_train, y_train)

    scores1.append(model1.score(X_test, y_test))
    scores2.append(model2.score(X_test, y_test))

print("Linear Regression Scores:", np.round(scores1, 4))
print("Decision Tree Scores:", np.round(scores2, 4))

diff = np.array(scores1) - np.array(scores2)
print("Difference:", np.round(diff, 4))

print("\n" + "="*50)


# 9. MODEL DRIFT SIMULATION

print("\n8. MODEL DRIFT SIMULATION")


# Use a reproducible split, independent of the paired-fold loop above.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model.fit(X_train, y_train)

# Synthetic sensitivity demo: noise scaled to each feature's training std dev.
# This is not a time-based evaluation or evidence of real-world model drift.
rng = np.random.default_rng(42)
noise_scale = 0.2 * X_train.std(axis=0)
X_new = X_test + rng.normal(size=X_test.shape) * noise_scale

print("Before Drift R2:", round(model.score(X_test, y_test), 4))
print("After Drift R2:", round(model.score(X_new, y_test), 4))

print("\n" + "="*50)


# 10. FINAL SUMMARY

print("\n9. FINAL SUMMARY")


print("""
- Train-Test Split scores vary with the split
- Cross Validation summarizes performance across multiple random folds
- R2 measures explained variance (higher is better)
- RMSE measures error (lower is better)
- Linear vs Tree shows model differences
- Paired CV ensures fair comparison
- Synthetic feature noise illustrates sensitivity, not measured drift over time
- Random folds assess this dataset; future sales require time-based validation
""")

Incomplete rows excluded: 2
Dataset: C:\Users\Administrator\Downloads\spark\day7\kc_house_data.csv
Target: price (USD)
Features: ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']

Dataset Info
Shape: (21611, 18)


1. TRAIN-TEST SPLIT (BASELINE)
R2 Score (Baseline): 0.7092


2. INSTABILITY OF TRAIN-TEST SPLIT
Run 1: R2 = 0.7003
Run 2: R2 = 0.7171
Run 3: R2 = 0.6946
Run 4: R2 = 0.6832
Run 5: R2 = 0.6981


3. K-FOLD CROSS VALIDATION (R2)
R2 Scores: [0.7092 0.6961 0.6862 0.698  0.6941]
Mean R2: 0.6967


4. CROSS VALIDATION (MSE & RMSE)


MSE Scores (USD squared): [4.34454195e+10 5.05659310e+10 3.75837858e+10 3.54494108e+10
 3.68922994e+10]
RMSE Scores (USD): [208435.6483 224868.6973 193865.3805 188280.1392 192073.6823]
Mean RMSE (USD): 201504.7095


5. SINGLE MODEL EVALUATION
Mean R2: 0.6967
Std Dev: 0.0074


6. MODEL COMPARISON


Linear Regression Mean R2: 0.6967
Decision Tree Mean R2: 0.7403


7. PAIRED K-FOLD COMPARISON
----------------------------


Linear Regression Scores: [0.7092 0.6961 0.6862 0.698  0.6941]
Decision Tree Scores: [0.727  0.7731 0.7289 0.7785 0.6939]
Difference: [-0.0178 -0.077  -0.0427 -0.0806  0.0002]


8. MODEL DRIFT SIMULATION
Before Drift R2: 0.7092
After Drift R2: 0.6967


9. FINAL SUMMARY

- Train-Test Split scores vary with the split
- Cross Validation summarizes performance across multiple random folds
- R2 measures explained variance (higher is better)
- RMSE measures error (lower is better)
- Linear vs Tree shows model differences
- Paired CV ensures fair comparison
- Synthetic feature noise illustrates sensitivity, not measured drift over time
- Random folds assess this dataset; future sales require time-based validation

